# Exploratory Data Analysis - Ferry Ticket Demand

This notebook performs exploratory data analysis on ferry ticket demand data for Toronto Island Park.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from preprocess import FerryDataPreprocessor, create_sample_data
from features import FerryFeatureEngineer

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully")

## 1. Load and Preprocess Data

In [ ]:
# Create sample data if not exists
sample_path = "../data/raw/ferry_sample_data.csv"
if not Path(sample_path).exists():
    create_sample_data(sample_path, days=30)
    print("Sample data created")

# Load and preprocess
preprocessor = FerryDataPreprocessor()
df = preprocessor.load_data(sample_path)
print(f"Original data shape: {df.shape}")
print(f"\nData columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Preprocess the data
df_processed = preprocessor.preprocess_pipeline(df)
print(f"Processed data shape: {df_processed.shape}")
df_processed.head()

## 2. Data Overview and Statistics

In [ ]:
# Basic statistics
print("=== Data Statistics ===")
print(df_processed.describe())

print("\n=== Data Info ===")
print(df_processed.info())

In [ ]:
# Check for missing values
print("Missing values:")
print(df_processed.isnull().sum())

## 3. Time Series Visualization

In [ ]:
# Plot sales over time
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_processed.index,
    y=df_processed['Sales Count'],
    mode='lines',
    name='Sales Count',
    line=dict(color='#1f77b4', width=1)
))

fig.update_layout(
    title='Ferry Ticket Sales Over Time',
    xaxis_title='Timestamp',
    yaxis_title='Sales Count',
    template='plotly_white',
    height=400
)

fig.show()

In [ ]:
# Plot sales and redemption together
fig = make_subplots(rows=2, cols=1, subplot_titles=('Sales Count', 'Redemption Count'),vertical_spacing=0.1)

fig.add_trace(go.Scatter(
    x=df_processed.index,
    y=df_processed['Sales Count'],
    mode='lines',
    name='Sales',
    line=dict(color='#1f77b4', width=1)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_processed.index,
    y=df_processed['Redemption Count'],
    mode='lines',
    name='Redemptions',
    line=dict(color='#ff7f0e', width=1)
), row=2, col=1)

fig.update_layout(
    title='Sales vs Redemption Counts',
    template='plotly_white',
    height=600
)

fig.show()

## 4. Temporal Patterns Analysis

In [ ]:
# Hourly pattern
df_processed['hour'] = df_processed.index.hour
hourly_sales = df_processed.groupby('hour')['Sales Count'].mean()

fig = go.Figure(data=[go.Bar(
    x=hourly_sales.index,
    y=hourly_sales.values,
    marker_color='steelblue'
)])

fig.update_layout(
    title='Average Hourly Sales Pattern',
    xaxis_title='Hour of Day',
    yaxis_title='Average Sales',
    template='plotly_white'
)

fig.show()

In [ ]:
# Day of week pattern
df_processed['day_of_week'] = df_processed.index.dayofweek
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_sales = df_processed.groupby('day_of_week')['Sales Count'].mean()

fig = go.Figure(data=[go.Bar(
    x=[day_names[i] for i in daily_sales.index],
    y=daily_sales.values,
    marker_color='coral'
)])

fig.update_layout(
    title='Average Sales by Day of Week',
    xaxis_title='Day of Week',
    yaxis_title='Average Sales',
    template='plotly_white'
)

fig.show()

In [ ]:
# Weekend vs Weekday
df_processed['is_weekend'] = df_processed.index.dayofweek >= 5
weekend_comparison = df_processed.groupby('is_weekend')['Sales Count'].mean()

fig = go.Figure(data=[go.Bar(
    x=['Weekday', 'Weekend'],
    y=[weekend_comparison[False], weekend_comparison[True]],
    marker_color=['lightblue', 'lightcoral']
)])

fig.update_layout(
    title='Weekday vs Weekend Sales',
    xaxis_title='Day Type',
    yaxis_title='Average Sales',
    template='plotly_white'
)

fig.show()

## 5. Distribution Analysis

In [ ]:
# Sales distribution
fig = go.Figure(data=[go.Histogram(
    x=df_processed['Sales Count'],
    nbinsx=50,
    marker_color='steelblue',
    opacity=0.7
)])

fig.update_layout(
    title='Distribution of Sales Count',
    xaxis_title='Sales Count',
    yaxis_title='Frequency',
    template='plotly_white'
)

fig.show()

In [ ]:
# Box plot by hour
fig = go.Figure()

for hour in range(24):
    hour_data = df_processed[df_processed['hour'] == hour]['Sales Count']
    fig.add_trace(go.Box(
        y=hour_data,
        name=f'{hour}:00',
        boxpoints='outliers'
    ))

fig.update_layout(
    title='Sales Distribution by Hour',
    xaxis_title='Hour',
    yaxis_title='Sales Count',
    template='plotly_white',
    height=500
)

fig.show()

## 6. Correlation Analysis

In [ ]:
# Correlation between sales and redemptions
correlation = df_processed[['Sales Count', 'Redemption Count']].corr()
print("Correlation Matrix:")
print(correlation)

# Scatter plot
fig = go.Figure(data=[go.Scatter(
    x=df_processed['Sales Count'],
    y=df_processed['Redemption Count'],
    mode='markers',
    marker=dict(
        color=df_processed['Sales Count'],
        colorscale='Viridis',
        showscale=True
    ),
    opacity=0.6
)])

fig.update_layout(
    title='Sales vs Redemption Scatter Plot',
    xaxis_title='Sales Count',
    yaxis_title='Redemption Count',
    template='plotly_white'
)

fig.show()

## 7. Feature Engineering Preview

In [ ]:
# Engineer features
engineer = FerryFeatureEngineer()
df_features = engineer.create_all_features(df_processed)

print(f"Feature matrix shape: {df_features.shape}")
print(f"\nFeature columns: {df_features.columns.tolist()}")
print(f"\nFirst few rows:")
df_features.head()

In [ ]:
# Display feature importance preview
print(f"Total features created: {len(engineer.feature_names)}")
print("\nFeature names:")
for i, feature in enumerate(engineer.feature_names, 1):
    print(f"{i}. {feature}")

## 8. Outlier Detection

In [ ]:
# Detect outliers using IQR method
df_with_outliers = preprocessor.detect_outliers(df_processed, method='iqr', threshold=3.0)

outlier_count = df_with_outliers['Sales Count_outlier'].sum()
print(f"Number of outliers detected: {outlier_count}")
print(f"Percentage of outliers: {outlier_count / len(df_processed) * 100:.2f}%")

In [ ]:
# Visualize outliers
fig = go.Figure()

# Normal points
normal_mask = ~df_with_outliers['Sales Count_outlier']
fig.add_trace(go.Scatter(
    x=df_with_outliers[normal_mask].index,
    y=df_with_outliers[normal_mask]['Sales Count'],
    mode='markers',
    name='Normal',
    marker=dict(color='blue', size=4)
))

# Outliers
outlier_mask = df_with_outliers['Sales Count_outlier']
fig.add_trace(go.Scatter(
    x=df_with_outliers[outlier_mask].index,
    y=df_with_outliers[outlier_mask]['Sales Count'],
    mode='markers',
    name='Outliers',
    marker=dict(color='red', size=8, symbol='x')
))

fig.update_layout(
    title='Outlier Detection - Sales Count',
    xaxis_title='Timestamp',
    yaxis_title='Sales Count',
    template='plotly_white',
    height=400
)

fig.show()

## 9. Summary and Key Findings

In [ ]:
print("=== EDA SUMMARY ===")
print(f"\nTotal records: {len(df_processed)}")
print(f"Date range: {df_processed.index.min()} to {df_processed.index.max()}")
print(f"Average sales: {df_processed['Sales Count'].mean():.2f}")
print(f"Average redemptions: {df_processed['Redemption Count'].mean():.2f}")
print(f"Peak sales hour: {df_processed.groupby('hour')['Sales Count'].mean().idxmax()}:00")
print(f"Peak sales day: {day_names[df_processed.groupby('day_of_week')['Sales Count'].mean().idxmax()]}")
print(f"Weekend vs Weekday ratio: {weekend_comparison[True] / weekend_comparison[False]:.2f}")
print(f"Outliers detected: {outlier_count}")

## 10. Save Processed Data

In [ ]:
# Save processed data for modeling
preprocessor.save_processed_data(df_processed, "../data/processed/ferry_processed.csv")
print("Processed data saved successfully")